# EduRecSys – Collaborative Filtering

This notebook implements a **Collaborative Filtering (CF)** recommendation system
based on user–item interaction data for the EduRecSys project.

Unlike content-based recommendation, collaborative filtering focuses on
**collective user behavior**, identifying patterns among users with similar
preferences to generate personalized recommendations.

---

## Objectives

The main objectives of this notebook are to:

- Simulate realistic user–course interaction data for educational content.
- Construct a user–item interaction matrix.
- Compute user similarity using cosine similarity.
- Generate personalized course recommendations using user-based collaborative filtering.

---

## Dataset Scope

Collaborative filtering is applied **only to Udemy (English) content** for the following reasons:

- Udemy courses are structured, well-defined items.
- They naturally align with user–item interaction modeling.
- SANAD Arabic articles lack explicit interaction signals suitable for CF.

---

## Important Note on Simulated Data

The Udemy dataset does not include explicit user ratings or interaction logs.
To demonstrate collaborative filtering concepts, **realistic user–course interactions
are simulated**.

This approach is commonly used in recommender system prototyping and academic demonstrations
when real interaction data is unavailable.

---

By the end of this notebook, the system will be able to recommend courses
based on **user similarity**, capturing collective preferences beyond textual similarity.


In [1]:
import pandas as pd
import numpy as np

content_en = pd.read_csv("/kaggle/input/edurecsys-processed-content/content_en.csv")
content_en.head()


,content_id,title,description,subject,level,content_duration,language
0,1070968,Ultimate Investment Banking Course,Ultimate Investment Banking Course,Business Finance,All Levels,1.5,en
1,1113822,Complete GST Course & Certification - Grow You...,Complete GST Course & Certification - Grow You...,Business Finance,All Levels,39.0,en
2,1006314,Financial Modeling for Business Analysts and C...,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level,2.5,en
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels,3.0,en
4,1011058,How To Maximize Your Profits Trading Options,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level,2.0,en


# User–Item Interaction Data

The Udemy dataset does not include explicit user ratings.
To enable collaborative filtering, we simulate realistic user–course interactions.

Each interaction represents a user enrolling in or rating a course.
This approach is commonly used in recommender system prototyping.


# Generate Users

In [2]:
np.random.seed(42)

num_users = 500
users = [f"user_{i}" for i in range(1, num_users + 1)]


# Generate Interactions

In [3]:
interactions = []

for user in users:
    sampled_courses = content_en.sample(
        np.random.randint(5, 20), replace=False
    )
    for _, row in sampled_courses.iterrows():
        interactions.append({
            "user_id": user,
            "content_id": row["content_id"],
            "rating": np.random.randint(3, 6)  # ratings between 3 and 5
        })

interactions_df = pd.DataFrame(interactions)
interactions_df.head()


,user_id,content_id,rating
0,user_1,71912,5
1,user_1,733654,4
2,user_1,200722,5
3,user_1,961996,5
4,user_1,775618,3


In [4]:
print(interactions_df.shape)
interactions_df["rating"].value_counts()


(5980, 3)


rating
4    2033
3    1987
5    1960
Name: count, dtype: int64

# Build User–Item Matrix

In [5]:
user_item_matrix = interactions_df.pivot_table(
    index="user_id",
    columns="content_id",
    values="rating"
).fillna(0)

user_item_matrix.head()


content_id,8325,11153,11174,11475,12214,12975,13216,14571,15285,15467,...,1269590,1270392,1271182,1271430,1272282,1274210,1275790,1275872,1276364,1277924
user_id,,,,,,,,,,,,,,,,,,,,,
user_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
user_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
user_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
user_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
user_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## User–Item Interaction Matrix

In this step, we constructed the user–item interaction matrix, which is the core representation used in collaborative filtering.

- Rows represent users.
- Columns represent educational content items.
- Cell values correspond to user ratings.
- Missing interactions were filled with zeros, indicating no interaction.

This matrix is typically sparse, as each user interacts with only a small subset of available courses. It will be used to compute user similarity in the next step.


# User Similarity (Cosine Similarity)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute user-user similarity matrix
user_similarity = cosine_similarity(user_item_matrix)

user_similarity.shape


(500, 500)

In [7]:
# Convert Similarity Matrix to DataFrame
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()


user_id,user_1,user_10,user_100,user_101,user_102,user_103,user_104,user_105,user_106,user_107,...,user_90,user_91,user_92,user_93,user_94,user_95,user_96,user_97,user_98,user_99
user_id,,,,,,,,,,,,,,,,,,,,,
user_1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.00000,0.087638,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
user_10,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
user_100,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.073159,0.0,0.000000,...,0.0,0.10499,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
user_101,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.0,0.065461,...,0.0,0.00000,0.000000,0.0,0.0,0.0,0.067106,0.0,0.0,0.0
user_102,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


# Finding Similar Users

In [8]:
def get_similar_users(user_id, top_k=5):
    """
    Returns top_k most similar users to the given user_id
    """
    # Similarity scores for the user
    sim_scores = user_similarity_df.loc[user_id]

    # Sort users by similarity (descending)
    sim_users = sim_scores.sort_values(ascending=False)

    # Exclude the user itself
    sim_users = sim_users.drop(user_id)

    return sim_users.head(top_k)


In [9]:
get_similar_users("user_1", top_k=5)

user_id
user_463    0.168750
user_333    0.141769
user_319    0.141696
user_2      0.135467
user_46     0.117037
Name: user_1, dtype: float64

For a given user, the system identifies the most similar users based on cosine similarity scores.
The user itself is excluded to avoid self-recommendation.
These similar users form the basis for collaborative recommendations.


# Collaborative Filtering Recommendation

In [10]:
def recommend_cf(user_id, top_k=5, n_similar_users=5):
    # Get similar users
    similar_users = get_similar_users(user_id, n_similar_users)

    # Courses already interacted by target user
    user_courses = interactions_df[
        interactions_df["user_id"] == user_id
    ]["content_id"].unique()

    # Interactions from similar users
    similar_interactions = interactions_df[
        interactions_df["user_id"].isin(similar_users.index)
    ]

    # Exclude already seen courses
    similar_interactions = similar_interactions[
        ~similar_interactions["content_id"].isin(user_courses)
    ]

    # Aggregate ratings
    recs = (
        similar_interactions
        .groupby("content_id")["rating"]
        .mean()
        .sort_values(ascending=False)
        .head(top_k)
        .reset_index()
    )

    # Merge with course metadata
    return recs.merge(content_en, on="content_id")[[
        "content_id", "title", "subject", "level", "rating"
    ]]


In [11]:
recommend_cf("user_1", top_k=5)

,content_id,title,subject,level,rating
0,153926,BITCOIN VISUALLY. Part I.,Business Finance,Beginner Level,5.0
1,286424,Learn to Design a Logo in Adobe Illustrator,Graphic Design,All Levels,5.0
2,861566,Instant harmonica - play Adele's wonderful 'He...,Musical Instruments,All Levels,5.0
3,672112,誰でもわかる Adobe Illustrator CS5,Graphic Design,Beginner Level,5.0
4,373636,Master the Trombone: Intermediate Instruction ...,Musical Instruments,All Levels,5.0


This function generates personalized recommendations using user-based collaborative filtering.

Steps:
1. Identify users with similar interaction patterns.
2. Collect courses rated by similar users.
3. Exclude courses already seen by the target user.
4. Rank remaining courses based on average rating.

This approach enables the system to capture collective user behavior and recommend content beyond textual similarity.


In [12]:
interactions_df.to_csv(
    "/kaggle/working/interactions.csv",
    index=False
)